# **Stage_07_07 -  Modelo TCN (many-to-one)**

**Introducción - Temporal Convolutional Networks (TCN)**

En esta notebook se introduce el uso de **Temporal Convolutional Networks (TCN)** como modelo base para la predicción de series temporales intradía.

Un TCN es una arquitectura neuronal basada en **convoluciones 1D causales y dilatadas**, diseñada para modelar secuencias temporales manteniendo el orden cronológico de la información. A diferencia de los modelos recurrentes, las TCN procesan la secuencia de forma completamente paralela, lo que mejora la estabilidad del entrenamiento y la eficiencia computacional.

Las principales características que motivan su uso en este proyecto son:

- Capacidad para capturar **dependencias temporales de corto y largo plazo**.
- **Causalidad estricta**, evitando cualquier fuga de información futura.
- Entrenamiento más **estable y reproducible** que modelos recurrentes clásicos.
- Buena adecuación a esquemas **seq2seq** para predicción multi-paso.

Por estas razones, el TCN se adopta como uno de los modelos principales para evaluar la capacidad predictiva sobre datos intradía del índice MNQ.



# **BLOQUE DE EJECUCIÓN COMPLETO**

## **1. Imports + paths**

In [ ]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


## **3. Rutas de ventanas seq2one y scalers**

In [ ]:
from pathlib import Path
import os

WINDOWS_SEQ2ONE_DIR = Path(
    os.environ.get("WINDOWS_SEQ2ONE_DIR", "data/windows/seq2one/")
)

SCALERS_DIR = Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

window_sizes = [30, 60, 90, 120, 180]
targets = ['delta_60', 'delta_90', 'ret_60', 'ret_90']
splits = ['train', 'valid', 'test']

In [ ]:
windows_paths = {}

for w in window_sizes:
    windows_paths[w] = {}

    for t in targets:
        windows_paths[w][t] = {}

        for s in splits:
            path = (
                DRIVE_DIR
                / WINDOWS_SEQ2ONE_DIR
                / f"L{w}"
                / f"windows_{t}_{s}.npz"
            )

            windows_paths[w][t][s] = path

#display(windows_paths)

#Como llamarlo:
#path_train_L60_delta = windows_paths[60]['delta_90']['train']
#print(path_train_L60_delta)

In [ ]:
scalers_paths = {}
for t in targets:
  scalers_paths[t] = {}
  path = (
                DRIVE_DIR
                / SCALERS_DIR
                / f"scaler_{t}.pkl"
            )

  scalers_paths[t] = path

display(scalers_paths)

{'delta_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl'),
 'delta_90': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_90.pkl'),
 'ret_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_ret_60.pkl'),
 'ret_90': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_ret_90.pkl')}

## **4. Reproducibilidad**

In [ ]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [ ]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [ ]:
print(compute_seq2one_metrics.__doc__)


    Calcula métricas simples y comparables para modelos seq2one.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    y_pred : np.ndarray
        Valores predichos con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    compute_r2 : bool
        Si True, calcula R² sobre el vector completo.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 en y_true o y_pred al calcular DA.
    allow_seq_inputs_take_last : bool
        Si True, permite inputs 2D (n_samples, seq_len) y toma el último paso [:, -1].
        Útil si algún modelo devuelve secuencia pero usted lo evalúa como many-to-one.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales.
    


## **6. Carga de data windows**

In [ ]:
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.
    """

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Usar contexto para cerrar correctamente el archivo
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # Opcional pero recomendable: copiar a memoria
        X = X.copy()
        y = y.copy()

    return X, y


In [ ]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [ ]:
from typing import Any, Dict, Mapping
from pathlib import Path

# --------------------------------------------------
# Carga completa: ventanas + scaler por window_size y target
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scalers_path: Mapping[str, Path],
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler correspondiente
    a un (window_size, target).

    windows_paths[L][target][split] -> Path
    scalers_path[target] -> Path
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    if target not in scalers_path:
        raise KeyError(f"target='{target}' no existe en scalers_path")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]
    scaler_path = scalers_paths[target]

    # --------------------------
    # 3) Carga
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test,  y_test  = load_npz_windows(test_path)

    scaler = load_scaler(scaler_path)

    # --------------------------
    # 4) Inferir horizonte
    # --------------------------
    horizon = int(target.split("_")[-1])

    # --------------------------
    # 5) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }

In [ ]:
import numpy as np

def maybe_flatten_X(X: np.ndarray, *, flatten: bool) -> np.ndarray:
    """
    Si flatten=True y X es 3D (N,L,F) -> (N, L*F)
    Si flatten=False -> retorna X tal cual.
    """
    if not flatten:
        return X
    if X.ndim == 3:
        N, L, F = X.shape
        return X.reshape(N, L * F)
    if X.ndim == 2:
        return X
    raise ValueError(f"X debe ser 2D o 3D, recibí shape={X.shape}")

In [ ]:
def create_bundles(window_size, targets: list, windows_paths=windows_paths, scalers_paths=scalers_paths, *, flatten_X=False):

    bundles = []
    for t in targets:
        b = load_windows_and_scaler(
            window_size=window_size,
            target=t,
            windows_paths=windows_paths,
            scalers_path=scalers_paths,
        )
        if flatten_X:
            b["train"]["X"] = maybe_flatten_X(b["train"]["X"], flatten=True)
            b["valid"]["X"] = maybe_flatten_X(b["valid"]["X"], flatten=True)
            b["test"]["X"]  = maybe_flatten_X(b["test"]["X"],  flatten=True)
        bundles.append(b)

    # prints (opcional)
    for b in bundles:
        print(f"H{b['horizon']} Train:", b["train"]["X"].shape, b["train"]["y"].shape)
        print(f"H{b['horizon']} Valid:", b["valid"]["X"].shape, b["valid"]["y"].shape)
        print(f"H{b['horizon']} Test :", b["test"]["X"].shape,  b["test"]["y"].shape)
        print(f"Scaler H{b['horizon']}:", type(b["scaler"]).__name__)

    return tuple(bundles)

In [ ]:
#bundle_delta_60, bundle_delta_90 = create_bundles(window_size = 30, targets = ['delta_60', 'delta_90'], windows_paths = windows_paths, scalers_paths = scalers_paths, flatten_X = False)
#bundle_ret_60, bundle_ret_90 = create_bundles(window_size = 30, targets = ['ret_60', 'ret_90'], windows_paths = windows_paths, scalers_paths = scalers_paths)
'''
def run_mlp(window_size: int, *, alpha: float = 1.0, verbose: bool = True):

    size = window_size

    if verbose:
        print("\n" + "=" * 80)
        print(f"RIDGE | SEQ2ONE | WINDOW_SIZE=L{size} | alpha={alpha}")
        print("=" * 80)

    targets = ["delta_60", "delta_90", "ret_60", "ret_90"]
    rows = []

    for target in targets:
        if verbose:
            print(f"\n[BUILD] L{size} | target = '{target}'")


        # crear SOLO 1 bundle (y aplanar X para Ridge)
        (bundle,) = create_bundles(
            window_size=size,
            targets=[target],            # <- SOLO UNO
            windows_paths=windows_paths,
            scalers_paths=scalers_paths,
            flatten_X=True,              # <- MLP necesita 2D
        )
'''


'\ndef run_mlp(window_size: int, *, alpha: float = 1.0, verbose: bool = True):\n\n    size = window_size\n\n    if verbose:\n        print("\n" + "=" * 80)\n        print(f"RIDGE | SEQ2ONE | WINDOW_SIZE=L{size} | alpha={alpha}")\n        print("=" * 80)\n\n    targets = ["delta_60", "delta_90", "ret_60", "ret_90"]\n    rows = []\n\n    for target in targets:\n        if verbose:\n            print(f"\n[BUILD] L{size} | target = \'{target}\'")\n\n\n        # crear SOLO 1 bundle (y aplanar X para Ridge)\n        (bundle,) = create_bundles(\n            window_size=size,\n            targets=[target],            # <- SOLO UNO\n            windows_paths=windows_paths,\n            scalers_paths=scalers_paths,\n            flatten_X=True,              # <- MLP necesita 2D\n        )\n'

NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **7. Sanity Check**

In [ ]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [ ]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [ ]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [ ]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

## **8. Métricas ML**


In [ ]:
import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    horizon: int,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])

## **9. Gestión de dataset de métricas**

In [ ]:
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [ ]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"seq2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

In [ ]:
import gc, torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gc.collect()
torch.cuda.empty_cache()

# **DEFINICIÓN DE MODELO**

## **7. Definición del modelo — placeholder**

### **7.1. Modelo TCN**

El **Temporal Convolutional Network (TCN)** es una arquitectura neuronal diseñada para modelar dependencias temporales en secuencias mediante **convoluciones 1D causales y dilatadas**, preservando estrictamente el orden temporal de la información.

A diferencia del MLP, el TCN no aplana la ventana histórica, sino que procesa la secuencia respetando su estructura temporal. En contraste con los modelos recurrentes (LSTM / GRU), el TCN no utiliza estados ocultos ni recurrencia explícita, sino que captura dependencias de largo alcance aumentando el **campo receptivo** a través de la dilatación.

En configuración **many-to-one**, el modelo utiliza la representación temporal final (último paso o pooling temporal) como resumen de toda la ventana para predecir un valor escalar futuro.

---

**Idea básica**

El TCN se construye a partir de bloques convolucionales causales, cada uno compuesto por:

- Convolución 1D causal dilatada  
- Activación no lineal  
- (Opcional) normalización  
- Conexión residual  

La dilatación permite que el modelo incorpore información de pasos temporales lejanos sin incrementar significativamente la profundidad.

Formalmente, una capa convolucional dilatada se define como:

$$
y_t = \sum_{k=0}^{K-1} w_k \, x_{t - d \cdot k}
$$

donde:

- $ x_t \in \mathbb{R}^{20} $ es el vector de features en el minuto $ t $,
- $ K $ es el tamaño del kernel,
- $ d $ es el factor de dilatación,
- la causalidad garantiza que $x_{t'}$ con $ t' > t $ no sea utilizada.

La salida final del bloque TCN, correspondiente al último instante temporal $ T $, se proyecta mediante una capa lineal:

$$
\hat{y} = W_o h_T + b_o
$$

donde $ h_T $ resume toda la ventana histórica (por ejemplo, 60 minutos).

---

**Regularización (TCN)**

Riesgo: **Medio**, controlado principalmente por la arquitectura.

Mecanismos principales:
- **Early stopping**: principal control de sobreajuste.
- **Profundidad moderada**: número limitado de bloques.
- **Dimensión de canales controlada**.
- **Dropout (opcional)**:
  - aplicado dentro de los bloques convolucionales,
  - no afecta la causalidad.

En general, el TCN presenta un entrenamiento más estable que los modelos recurrentes y requiere menos ajustes finos de regularización.

---

**Por qué el TCN es relevante en este proyecto**

- Entrada secuencial explícita: **60 x 20**.
- Capacidad para capturar:
  - dependencias temporales de corto, mediano y largo plazo,
  - estructura intradía sin recurrencia.
- Modelo:
  - completamente paralelizable,
  - estable en entrenamiento,
  - adecuado para esquemas **seq2seq** y **many-to-one**.

El TCN actúa como una alternativa no recurrente robusta frente a LSTM y GRU, permitiendo evaluar si una arquitectura convolucional logra un mejor compromiso entre desempeño, estabilidad y eficiencia.

---

**Hiperparámetros iniciales**

Para este stage (sin tuning):

- Tipo: TCN many-to-one  
- Número de bloques: bajo (por ejemplo, 3–5)  
- Tamaño del kernel: pequeño (2–5)  
- Canales por bloque: moderados  
- Dilataciones: crecientes (potencias de 2)  
- Dropout: desactivado inicialmente  
- Optimización: Adam  
- Early stopping: activado  
- Evaluación externa sobre VALID  

El ajuste fino de la arquitectura se aborda en etapas posteriores.


A continuación tiene una implementación TCN many-to-one con tuning de hiperparámetros, siguiendo una estructura típica y reutilizable (dataset por bundle, entrenamiento, validación externa, Optuna).

Supuesto razonable (alineado a notebooks anteriores): ya dispone de bundle con `X_train`, `y_train`, `X_valid`, `y_valid` (y opcional `X_test`, `y_test`), donde:

- `X` tiene shape (`N`, `seq_len`, `n_features`)
- `y` tiene shape (`N`,) o (`N`, `1`)

### **7.2. Imports (PyTorch) + semillas**

In [ ]:
from __future__ import annotations

import math
import time
from dataclasses import dataclass
from typing import Dict, Any, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader


### **7.3. Utilidades**

In [ ]:
def set_seed(seed: int = 42) -> None:
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def to_1d(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y)
    if y.ndim == 2 and y.shape[1] == 1:
        return y[:, 0]
    return y


### **7.4. DataLoaders desde bundle (con reshape interno)**

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

def make_loaders_from_bundle_3d(
    bundle: dict,
    *,
    seq_len: int | None = None,
    n_features: int | None = None,
    batch_size: int = 16384,
    num_workers: int = 2,
) -> dict:
    """
    Crea loaders train/valid/test para LSTM many-to-one.

    Espera:
      - X: (n, seq_len, n_features)  (ya 3D)
      - y: (n,) o (n,1)  -> (n,1)

    Si seq_len/n_features se pasan, valida consistencia.
    """
    loaders = {}

    for split in ["train", "valid", "test"]:
        X = np.asarray(bundle[split]["X"], dtype=np.float32)
        y = np.asarray(bundle[split]["y"], dtype=np.float32).reshape(-1, 1)

        if X.ndim != 3:
            raise ValueError(
                f"[{split}] Se esperaba X 3D (n, seq_len, n_features). "
                f"Recibido shape={X.shape} (ndim={X.ndim})."
            )

        n, sl, nf = X.shape

        if seq_len is not None and sl != int(seq_len):
            raise ValueError(f"[{split}] seq_len esperado={seq_len}, recibido={sl}. shape={X.shape}")

        if n_features is not None and nf != int(n_features):
            raise ValueError(f"[{split}] n_features esperado={n_features}, recibido={nf}. shape={X.shape}")

        if y.shape[0] != n:
            raise ValueError(f"[{split}] X e y no alinean: X n={n}, y n={y.shape[0]}.")

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        shuffle = (split == "train")

        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
        )

    return loaders

### **7.5. TCN core: TemporalBlock + TCNRegressor (many-to-one)**

In [ ]:
class Chomp1d(nn.Module):
    """
    Recorta padding agregado por Conv1d para mantener causalidad.
    Elimina elementos a la derecha (futuro).
    """
    def __init__(self, chomp_size: int):
        super().__init__()
        self.chomp_size = int(chomp_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, T)
        if self.chomp_size == 0:
            return x
        # elimina padding "futuro"
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    """
    Bloque residual dilatado del TCN.

    - Conv1D dilatada causal
    - ReLU + Dropout
    - Residual connection
    """
    def __init__(
        self,
        in_ch: int,
        out_ch: int,
        *,
        kernel_size: int,
        dilation: int,
        dropout: float,
    ):
        super().__init__()

        k = int(kernel_size)
        d = int(dilation)

        # padding necesario para mantener longitud temporal
        # se elimina luego con Chomp1d para garantizar causalidad
        pad = (k - 1) * d

        # Primera convolución dilatada
        self.conv1 = nn.Conv1d(
            in_ch,
            out_ch,
            kernel_size=k,
            dilation=d,
            padding=pad,
        )
        self.chomp1 = Chomp1d(pad)
        self.act1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)

        # Segunda convolución dilatada
        self.conv2 = nn.Conv1d(
            out_ch,
            out_ch,
            kernel_size=k,
            dilation=d,
            padding=pad,
        )
        self.chomp2 = Chomp1d(pad)
        self.act2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout)

        # Ajuste dimensional para residual si cambian canales
        self.downsample = (
            nn.Conv1d(in_ch, out_ch, kernel_size=1)
            if in_ch != out_ch
            else None
        )

        self.out_act = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, T)

        y = self.conv1(x)
        y = self.chomp1(y)
        y = self.act1(y)
        y = self.drop1(y)

        y = self.conv2(y)
        y = self.chomp2(y)
        y = self.act2(y)
        y = self.drop2(y)

        # conexión residual
        res = x if self.downsample is None else self.downsample(x)

        return self.out_act(y + res)


class TCNRegressor(nn.Module):
    """
    TCN many-to-one (seq2one).

    Entrada:
        X: (B, T, F)

    Salida:
        y: (B,)
    """
    def __init__(
        self,
        n_features: int,
        *,
        hidden_dim: int,
        num_layers: int,
        kernel_size: int,
        dropout: float,
        use_layernorm: bool = False,
    ):
        super().__init__()

        self.n_features = int(n_features)
        self.hidden_dim = int(hidden_dim)
        self.num_layers = int(num_layers)

        blocks = []
        in_ch = self.n_features

        # Construcción automática de capas con mismo hidden_dim
        # dilations exponenciales: 1, 2, 4, 8, ...
        for i in range(self.num_layers):
            dilation = 2 ** i

            blocks.append(
                TemporalBlock(
                    in_ch,
                    self.hidden_dim,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )

            in_ch = self.hidden_dim

        self.tcn = nn.Sequential(*blocks)

        # LayerNorm opcional sobre último estado
        self.use_layernorm = bool(use_layernorm)
        self.ln = nn.LayerNorm(self.hidden_dim) if self.use_layernorm else None

        # Capa final de regresión
        self.head = nn.Linear(self.hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F)
        # Conv1d espera (B, C, T)
        x = x.transpose(1, 2)  # → (B, F, T)

        h = self.tcn(x)        # → (B, hidden_dim, T)

        # many-to-one: usamos último timestep
        h_last = h[:, :, -1]   # → (B, hidden_dim)

        if self.ln is not None:
            h_last = self.ln(h_last)

        y = self.head(h_last).squeeze(-1)  # → (B,)

        return y


### **7.6. Train / eval loop (Early stopping)**



In [ ]:
# ============================================================
# 2) Train / eval loop (Early stopping)
# ============================================================

@dataclass
class TrainConfig:
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    epochs: int = 30
    patience: int = 6
    grad_clip: float = 1.0
    log_every: int = 200


@torch.no_grad()
def evaluate_mae_rmse(model: nn.Module, dl: DataLoader, device: str) -> Dict[str, float]:
    model.eval()
    y_true_all, y_pred_all = [], []

    for xb, yb in dl:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        # Asegura shape compatible con salida del modelo (B,)
        yb = yb.view(-1)

        yp = model(xb).view(-1)

        y_true_all.append(yb.detach().cpu().numpy())
        y_pred_all.append(yp.detach().cpu().numpy())

    y_true = np.concatenate(y_true_all)
    y_pred = np.concatenate(y_pred_all)

    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    return {"MAE": mae, "RMSE": rmse}


def train_one_trial(
    model: nn.Module,
    dl_train: DataLoader,
    dl_valid: DataLoader,
    *,
    lr: float,
    weight_decay: float,
    cfg: TrainConfig,
) -> Dict[str, Any]:
    device = cfg.device
    model.to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_mae = float("inf")
    best_state = None
    bad = 0

    step = 0
    t0 = time.time()

    for ep in range(cfg.epochs):
        model.train()

        for xb, yb in dl_train:
            step += 1
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            # Asegura shape compatible con salida del modelo (B,)
            yb = yb.view(-1)

            opt.zero_grad(set_to_none=True)

            yp = model(xb).view(-1)

            # Huber/SmoothL1: robusto a outliers
            loss = F.smooth_l1_loss(yp, yb)

            loss.backward()

            if cfg.grad_clip is not None and cfg.grad_clip > 0:
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

            opt.step()

        metrics = evaluate_mae_rmse(model, dl_valid, device)
        mae = metrics["MAE"]

        if mae < best_mae:
            best_mae = mae
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= cfg.patience:
                break

    elapsed = time.time() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    out = {
        "best_valid_MAE": float(best_mae),
        "final_valid": evaluate_mae_rmse(model, dl_valid, device),
        "elapsed_sec": float(elapsed),
        "epochs_ran": ep + 1,
        "best_state_dict": best_state,  # opcional: guardar afuera
    }
    return out



### **7.8. Run tuning**


In [ ]:
set_seed(42)

CFG = TrainConfig(
    device="cuda" if torch.cuda.is_available() else "cpu",
    epochs=12,        # antes 30
    patience=3,       # antes 6
    grad_clip=1.0,
)


### **7.9. Ejecución**


In [ ]:
import torch

# ------------------------------------------------------------
# Check CUDA
# ------------------------------------------------------------
assert torch.cuda.is_available(), "CUDA no disponible"

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

# ------------------------------------------------------------
# Check modelo en GPU
# ------------------------------------------------------------

n_features = 36

model = TCNRegressor(
    n_features=n_features,
    hidden_dim=32,
    num_layers=3,
    kernel_size=3,
    dropout=0.0,
)

model.to(CFG.device)

assert next(model.parameters()).is_cuda, "Modelo NO está en GPU"

print("Modelo en GPU: OK")


GPU: NVIDIA L4
CUDA: 12.6
Modelo en GPU: OK


### **7.10 Entrenamiento TCN con best params**


In [ ]:
def train_tcn(
    loaders: dict,
    *,
    n_features: int,
    device: torch.device,
    hidden_dim: int = 64,
    num_layers: int = 3,
    kernel_size: int = 3,
    dropout: float = 0.0,
    use_layernorm: bool = False,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = True,
) -> tuple[nn.Module, dict]:
    """
    Entrena TCN seq2one con early stopping por valid_mse, igual estilo train_gru.

    loaders: dict con keys {"train","valid","test"} y batches (xb, yb)
    - xb: (B, L, F)
    - yb: (B, 1) o (B,)  (se normaliza internamente)
    """
    model = TCNRegressor(
        n_features=n_features,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        kernel_size=kernel_size,
        dropout=dropout,
        use_layernorm=use_layernorm,
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    scheduler = None
    if use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode="min", factor=0.5, patience=2, min_lr=1e-5
        )

    best_state = None
    best_valid = float("inf")
    bad_epochs = 0

    history = {"best_valid_mse": None, "epochs_ran": 0, "final_lr": None}

    for epoch in range(1, max_epochs + 1):
        model.train()

        for xb, yb in loaders["train"]:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            # Normaliza shapes para evitar broadcasting accidental:
            # - pred: (B,)   -> (B,1)
            # - yb:   (B,) o (B,1) -> (B,1)
            yb = yb.view(-1, 1)

            opt.zero_grad(set_to_none=True)
            pred = model(xb).view(-1, 1)
            loss = loss_fn(pred, yb)
            loss.backward()

            if clip_grad_norm is not None:
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(clip_grad_norm))

            opt.step()

        # Reusa tu eval_mse (misma lógica que GRU)
        valid_mse = eval_mse(model, loaders["valid"], device)
        if scheduler is not None:
            scheduler.step(valid_mse)

        current_lr = opt.param_groups[0]["lr"]
        print(f"epoch={epoch:02d} | valid_mse={valid_mse:.6f} | lr={current_lr:.2e}")

        history["epochs_ran"] = epoch
        history["final_lr"] = float(current_lr)

        if valid_mse < best_valid - 1e-9:
            best_valid = valid_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping (patience={patience}). Best valid_mse={best_valid:.6f}")
                break

    history["best_valid_mse"] = float(best_valid)
    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history



H60 -> n_features: 20
H60 train X: (330144, 1200) y: (330144,)
H60 valid X: (70952, 1200) y: (70952,)
H60 test  X: (70590, 1200) y: (70590,)

H90 -> n_features: 20
H90 train X: (330144, 1200) y: (330144,)
H90 valid X: (70952, 1200) y: (70952,)
H90 test  X: (70590, 1200) y: (70590,)


In [ ]:
@torch.no_grad()
def predict_tcn(
    model: nn.Module,
    X_seq: np.ndarray,
    *,
    device: torch.device,
    batch_size: int = 4096,
) -> np.ndarray:
    """
    Predicción para TCN many-to-one (seq2one).

    Espera:
        X_seq: (n, L, F)
    Retorna:
        preds: (n,)
    """
    model.eval()

    X_seq = np.asarray(X_seq, dtype=np.float32)
    if X_seq.ndim != 3:
        raise ValueError(f"Se esperaba X 3D (n, L, F). Recibido shape={X_seq.shape}")

    n = X_seq.shape[0]
    preds = []

    for i in range(0, n, batch_size):
        xb = torch.from_numpy(X_seq[i:i+batch_size]).to(device, non_blocking=True)
        yb = model(xb).view(-1)  # (B,)
        preds.append(yb.detach().cpu().numpy())
        del xb, yb

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return np.concatenate(preds, axis=0)


In [ ]:
@torch.no_grad()
def eval_mse(model: nn.Module, dl: DataLoader, device: torch.device) -> float:
    """
    Retorna MSE sobre dl. Compatible con modelos que devuelven (B,) o (B,1),
    y con yb en dl como (B,1).
    """
    model.eval()
    mse_sum = 0.0
    n = 0

    for xb, yb in dl:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True).view(-1)   # (B,)

        yp = model(xb).view(-1)                          # (B,)

        mse_sum += torch.sum((yp - yb) ** 2).item()
        n += yb.numel()

    return mse_sum / max(n, 1)


In [ ]:
def train_tcn(
    loaders: dict,
    *,
    n_features: int,
    device: torch.device,
    hidden_dim: int = 64,
    num_layers: int = 3,
    kernel_size: int = 3,
    dropout: float = 0.0,
    use_layernorm: bool = False,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = True,
) -> tuple[nn.Module, dict]:
    """
    Entrena TCN seq2one (many-to-one) con early stopping por valid_mse.
    Mantiene la misma firma y contrato que train_gru.
    """

    model = TCNRegressor(
        n_features=n_features,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        kernel_size=kernel_size,
        dropout=dropout,
        use_layernorm=use_layernorm,
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    scheduler = None
    if use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode="min", factor=0.5, patience=2, min_lr=1e-5
        )

    best_state = None
    best_valid = float("inf")
    bad_epochs = 0

    history = {"best_valid_mse": None, "epochs_ran": 0, "final_lr": None}

    for epoch in range(1, max_epochs + 1):
        model.train()

        for xb, yb in loaders["train"]:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True).view(-1, 1)  # (B,1)

            opt.zero_grad(set_to_none=True)

            pred = model(xb).view(-1, 1)  # (B,1)
            loss = loss_fn(pred, yb)
            loss.backward()

            if clip_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(clip_grad_norm))

            opt.step()

        valid_mse = eval_mse(model, loaders["valid"], device)
        if scheduler is not None:
            scheduler.step(valid_mse)

        current_lr = opt.param_groups[0]["lr"]
        print(f"epoch={epoch:02d} | valid_mse={valid_mse:.6f} | lr={current_lr:.2e}")

        history["epochs_ran"] = epoch
        history["final_lr"] = float(current_lr)

        if valid_mse < best_valid - 1e-9:
            best_valid = valid_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping (patience={patience}). Best valid_mse={best_valid:.6f}")
                break

    history["best_valid_mse"] = float(best_valid)
    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history


In [ ]:
import pandas as pd

def run_gru_incremental(
    window_sizes: list[int],
    *,
    hidden_size: int = 64,
    num_layers: int = 1,
    dropout: float = 0.0,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = True,
    name: str = "gru",
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Corre GRU seq2one incremental por window_size, guardando checkpoints.
    """

    df_all = load_seq2one_metrics_if_exists(name=name, base_dir=base_dir)

    base_cols = ["model","split","window_size","target","horizon_min","MAE","RMSE","R2","DA"]
    hp_cols = ["hidden_size","num_layers","dropout","lr","w_decay","max_epochs","patience","clip_grad_norm","use_scheduler"]

    if df_all.empty:
        df_all = pd.DataFrame(columns=base_cols + hp_cols)

    for c in base_cols + hp_cols:
        if c not in df_all.columns:
            df_all[c] = pd.NA

    key_cols = [
        "model",
        "hidden_size","num_layers","dropout","lr","w_decay",
        "max_epochs","patience","clip_grad_norm","use_scheduler",
        "window_size","target","split","horizon_min"
    ]

    if len(df_all):
        df_all["window_size"] = pd.to_numeric(df_all["window_size"], errors="coerce").astype("Int64")
        df_all["horizon_min"] = pd.to_numeric(df_all["horizon_min"], errors="coerce").astype("Int64")

    for ws in window_sizes:

        df_ws = df_all[
            (df_all["model"] == "gru") &
            (df_all["hidden_size"] == hidden_size) &
            (df_all["num_layers"] == num_layers) &
            (df_all["dropout"] == dropout) &
            (df_all["lr"] == lr) &
            (df_all["w_decay"] == weight_decay) &
            (df_all["max_epochs"] == max_epochs) &
            (df_all["patience"] == patience) &
            (df_all["clip_grad_norm"] == clip_grad_norm) &
            (df_all["use_scheduler"] == use_scheduler) &
            (df_all["window_size"] == ws)
        ]

        if len(df_ws) >= 8:
            if verbose:
                print(f"[SKIP] L{ws}: ya hay {len(df_ws)} filas (gru hs={hidden_size} w_decay={weight_decay}).")
            continue

        if verbose:
            print("\n" + "="*90)
            print(
                f"[RUN] GRU incremental | L{ws} | (L,F)=({ws},36) "
                f"| hs={hidden_size} | Ls={num_layers} | do={dropout} | lr={lr} | w_decay={weight_decay} "
                f"| ep={max_epochs} | pat={patience} | clip={clip_grad_norm} | sched={use_scheduler}"
            )
            print("="*90)

        df_new = run_gru(
            ws,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            lr=lr,
            weight_decay=weight_decay,
            max_epochs=max_epochs,
            patience=patience,
            clip_grad_norm=clip_grad_norm,
            use_scheduler=use_scheduler,
            verbose=verbose,
        ).copy()

        df_new["hidden_size"] = hidden_size
        df_new["num_layers"] = num_layers
        df_new["dropout"] = dropout
        df_new["lr"] = lr
        df_new["w_decay"] = weight_decay
        df_new["max_epochs"] = max_epochs
        df_new["patience"] = patience
        df_new["clip_grad_norm"] = clip_grad_norm
        df_new["use_scheduler"] = use_scheduler

        existing_keys = set(tuple(x) for x in df_all[key_cols].dropna().values)
        mask_keep = [tuple(row) not in existing_keys for row in df_new[key_cols].values]
        df_new = df_new.loc[mask_keep].copy()

        if df_new.empty:
            if verbose:
                print(f"[INFO] L{ws}: no había filas nuevas para agregar.")
            continue

        df_all = pd.concat([df_all, df_new], ignore_index=True)
        df_all = df_all.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)

        save_seq2one_metrics(df_all, name=name, base_dir=base_dir)

        if verbose:
            print(f"[OK] Checkpoint guardado. Total rows={len(df_all)}")

    return df_all

In [ ]:
import pandas as pd
import numpy as np
import torch
import gc
import time

def _ts():
    return time.strftime("%H:%M:%S")


def run_tcn(
    window_size: int,
    *,
    n_features: int = 36,
    batch_size_train: int = 4096,
    batch_size_pred: int = 2048,
    hidden_dim: int = 64,
    num_layers: int = 3,
    kernel_size: int = 3,
    dropout: float = 0.0,
    use_layernorm: bool = False,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = True,
    verbose: bool = True,
):
    L = int(window_size)

    if verbose:
        print("\n" + "=" * 80)
        print(
            f"[{_ts()}] TCN | SEQ2ONE | WINDOW_SIZE=L{L} | (L,F)=({L},{n_features}) "
            f"| hd={hidden_dim} | nL={num_layers} | k={kernel_size} | do={dropout} | ln={use_layernorm} "
            f"| wd={weight_decay}"
        )
        print("=" * 80)

    targets = ["delta_60", "delta_90", "ret_60", "ret_90"]
    rows = []
    t_global = time.perf_counter()

    for i, target in enumerate(targets, start=1):
        t_target = time.perf_counter()

        if verbose:
            print(f"\n[{_ts()}] [{i}/{len(targets)}] START target='{target}' | L{L}")

        bundle = None
        loaders = None
        model = None
        hist = None
        metrics_valid = None
        metrics_test = None

        try:
            # -------------------------
            # BUILD BUNDLE (3D)
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [BUILD] Creando bundle (flatten_X=False) ...")
            t0 = time.perf_counter()

            (bundle,) = create_bundles(
                window_size=L,
                targets=[target],
                windows_paths=windows_paths,
                scalers_paths=scalers_paths,
                flatten_X=False,  # TCN necesita 3D
            )

            if verbose:
                dt = time.perf_counter() - t0
                try:
                    xshape = bundle["train"]["X"].shape
                    yshape = bundle["train"]["y"].shape
                    print(f"[{_ts()}]   [BUILD] OK | train X={xshape} y={yshape} | dt={dt:.2f}s")
                except Exception:
                    print(f"[{_ts()}]   [BUILD] OK | dt={dt:.2f}s")

            # -------------------------
            # LOADERS (3D estrictos)
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [LOADERS] Creando DataLoaders (3D) ...")
            t0 = time.perf_counter()

            loaders = make_loaders_from_bundle_3d(
                bundle,
                seq_len=L,
                n_features=n_features,
                batch_size=batch_size_train,
            )

            if verbose:
                dt = time.perf_counter() - t0
                try:
                    ntr = len(loaders["train"].dataset)
                    nva = len(loaders["valid"].dataset)
                    nte = len(loaders["test"].dataset)
                    print(f"[{_ts()}]   [LOADERS] OK | n(train/valid/test)=({ntr}/{nva}/{nte}) | dt={dt:.2f}s")
                except Exception:
                    print(f"[{_ts()}]   [LOADERS] OK | dt={dt:.2f}s")

            # -------------------------
            # TRAIN
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [TRAIN] Iniciando entrenamiento ...")
            t0 = time.perf_counter()

            model, hist = train_tcn(
                loaders,
                n_features=n_features,
                hidden_dim=hidden_dim,
                num_layers=num_layers,
                kernel_size=kernel_size,
                dropout=dropout,
                use_layernorm=use_layernorm,
                lr=lr,
                weight_decay=weight_decay,
                max_epochs=max_epochs,
                patience=patience,
                clip_grad_norm=clip_grad_norm,
                use_scheduler=use_scheduler,
                device=device,
            )

            if verbose:
                dt = time.perf_counter() - t0
                print(f"[{_ts()}]   [TRAIN] FIN entrenamiento | dt={dt:.2f}s")

            # liberar TRAIN (opcional)
            if verbose:
                print(f"[{_ts()}]   [MEM] Liberando bundle['train'] y gc.collect() ...")
            del bundle["train"]
            gc.collect()

            # -------------------------
            # PRED + METRICS
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [PRED] Predicciones + métricas (valid/test) ...")
            t0 = time.perf_counter()

            metrics_valid, metrics_test = get_metrics_torch(
                bundle,
                model,
                device=device,
                predict_fn=predict_tcn,          # <- TCN
                batch_size_pred=batch_size_pred,
                compute_r2=True,
            )

            if verbose:
                dt = time.perf_counter() - t0
                print(f"[{_ts()}]   [METRICS] OK (valid/test) | dt={dt:.2f}s")

            # -------------------------
            # DF APPEND
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [DF] Agregando filas a la tabla ...")
            t0 = time.perf_counter()

            df_v = metrics_to_df(
                metrics_valid,
                model="tcn",
                split="valid",
                horizon=bundle["horizon"],
                window_size=bundle["window_size"],
                target=bundle["target"],
            )

            df_t = metrics_to_df(
                metrics_test,
                model="tcn",
                split="test",
                horizon=bundle["horizon"],
                window_size=bundle["window_size"],
                target=bundle["target"],
            )

            for df_ in (df_v, df_t):
                df_["hidden_dim"] = hidden_dim
                df_["num_layers"] = num_layers
                df_["kernel_size"] = kernel_size
                df_["dropout"] = dropout
                df_["use_layernorm"] = use_layernorm
                df_["lr"] = lr
                df_["weight_decay"] = weight_decay
                if isinstance(hist, dict):
                    df_["best_valid_mse"] = hist.get("best_valid_mse")
                    df_["epochs_ran"] = hist.get("epochs_ran")
                    df_["final_lr"] = hist.get("final_lr")

            rows.append(df_v)
            rows.append(df_t)

            if verbose:
                dt = time.perf_counter() - t0
                print(f"[{_ts()}]   [DF] OK | dt={dt:.2f}s")

            if verbose:
                dt_target = time.perf_counter() - t_target
                print(f"[{_ts()}] [{i}/{len(targets)}] DONE target='{target}' | dt_total={dt_target:.2f}s")

        finally:
            if verbose:
                print(f"[{_ts()}]   [CLEAN] Liberando objetos ...")

            bundle = loaders = model = hist = metrics_valid = metrics_test = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # -------------------------
    # FINAL DF
    # -------------------------
    if verbose:
        print(f"\n[{_ts()}] [FINAL] Concatenando resultados ...")
    t0 = time.perf_counter()

    df_tcn_metrics = (
        pd.concat(rows, ignore_index=True)
          .sort_values(["window_size", "target", "split", "horizon_min", "model"])
          .reset_index(drop=True)
    )

    if verbose:
        dt = time.perf_counter() - t0
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [FINAL] OK | rows={len(df_tcn_metrics)} | dt_concat={dt:.2f}s | dt_total={dt_all:.2f}s")
        print(df_tcn_metrics[["window_size", "target", "split", "horizon_min", "model"]]
              .drop_duplicates()
              .to_string(index=False))

    return df_tcn_metrics


In [ ]:
import pandas as pd

def run_tcn_incremental(
    window_sizes: list[int],
    *,
    hidden_dim: int = 64,
    num_layers: int = 3,
    kernel_size: int = 3,
    dropout: float = 0.0,
    use_layernorm: bool = False,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = True,
    name: str = "tcn",
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Corre TCN seq2one incremental por window_size, guardando checkpoints.
    """

    df_all = load_seq2one_metrics_if_exists(name=name, base_dir=base_dir)

    base_cols = ["model","split","window_size","target","horizon_min","MAE","RMSE","R2","DA"]
    hp_cols = [
        "hidden_dim","num_layers","kernel_size","dropout","use_layernorm",
        "lr","w_decay","max_epochs","patience","clip_grad_norm","use_scheduler"
    ]

    if df_all.empty:
        df_all = pd.DataFrame(columns=base_cols + hp_cols)

    for c in base_cols + hp_cols:
        if c not in df_all.columns:
            df_all[c] = pd.NA

    key_cols = [
        "model",
        "hidden_dim","num_layers","kernel_size","dropout","use_layernorm","lr","w_decay",
        "max_epochs","patience","clip_grad_norm","use_scheduler",
        "window_size","target","split","horizon_min"
    ]

    if len(df_all):
        df_all["window_size"] = pd.to_numeric(df_all["window_size"], errors="coerce").astype("Int64")
        df_all["horizon_min"] = pd.to_numeric(df_all["horizon_min"], errors="coerce").astype("Int64")

    for ws in window_sizes:

        df_ws = df_all[
            (df_all["model"] == "tcn") &
            (df_all["hidden_dim"] == hidden_dim) &
            (df_all["num_layers"] == num_layers) &
            (df_all["kernel_size"] == kernel_size) &
            (df_all["dropout"] == dropout) &
            (df_all["use_layernorm"] == use_layernorm) &
            (df_all["lr"] == lr) &
            (df_all["w_decay"] == weight_decay) &
            (df_all["max_epochs"] == max_epochs) &
            (df_all["patience"] == patience) &
            (df_all["clip_grad_norm"] == clip_grad_norm) &
            (df_all["use_scheduler"] == use_scheduler) &
            (df_all["window_size"] == ws)
        ]

        if len(df_ws) >= 8:
            if verbose:
                print(f"[SKIP] L{ws}: ya hay {len(df_ws)} filas (tcn hd={hidden_dim} w_decay={weight_decay}).")
            continue

        if verbose:
            print("\n" + "="*90)
            print(
                f"[RUN] TCN incremental | L{ws} | (L,F)=({ws},36) "
                f"| hd={hidden_dim} | nL={num_layers} | k={kernel_size} | do={dropout} | ln={use_layernorm} "
                f"| lr={lr} | w_decay={weight_decay} | ep={max_epochs} | pat={patience} "
                f"| clip={clip_grad_norm} | sched={use_scheduler}"
            )
            print("="*90)

        df_new = run_tcn(
            ws,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            kernel_size=kernel_size,
            dropout=dropout,
            use_layernorm=use_layernorm,
            lr=lr,
            weight_decay=weight_decay,
            max_epochs=max_epochs,
            patience=patience,
            clip_grad_norm=clip_grad_norm,
            use_scheduler=use_scheduler,
            verbose=verbose,
        ).copy()

        # columna estándar usada en incremental (w_decay)
        df_new["hidden_dim"] = hidden_dim
        df_new["num_layers"] = num_layers
        df_new["kernel_size"] = kernel_size
        df_new["dropout"] = dropout
        df_new["use_layernorm"] = use_layernorm
        df_new["lr"] = lr
        df_new["w_decay"] = weight_decay
        df_new["max_epochs"] = max_epochs
        df_new["patience"] = patience
        df_new["clip_grad_norm"] = clip_grad_norm
        df_new["use_scheduler"] = use_scheduler

        existing_keys = set(tuple(x) for x in df_all[key_cols].dropna().values)
        mask_keep = [tuple(row) not in existing_keys for row in df_new[key_cols].values]
        df_new = df_new.loc[mask_keep].copy()

        if df_new.empty:
            if verbose:
                print(f"[INFO] L{ws}: no había filas nuevas para agregar.")
            continue

        df_all = pd.concat([df_all, df_new], ignore_index=True)
        df_all = df_all.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)

        save_seq2one_metrics(df_all, name=name, base_dir=base_dir)

        if verbose:
            print(f"[OK] Checkpoint guardado. Total rows={len(df_all)}")

    return df_all


In [ ]:
df_tcn_all_sizes = run_tcn_incremental(
    window_sizes=[30, 60, 90, 120, 180],
    hidden_dim=64,
    num_layers=3,
    kernel_size=3,
    dropout=0.0,
    use_layernorm=False,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=30,
    patience=5,
    clip_grad_norm=1.0,
    use_scheduler=True,
    name="tcn",
    verbose=True,
)
